# Fine-tune `uae-finance-bot` on a free Colab GPU (QLoRA)

This teaches Llama-3.2-3B-Instruct to (a) call `render_chart` / `compound_interest_calculator` with correct arguments when a request needs them, and (b) answer ordinary questions in plain text. It trains on the **exact prompts your local Ollama chatbot sends** -- the real system prompt, real retrieved passages, real tool definitions -- built by `finetune/build_training_data.py`, so what it learns transfers to the app.

**Why this fits a free T4:** the model is loaded in 4-bit (QLoRA) and trained in fp16 (a T4 has no bf16). Expect roughly 8-10 GB of the T4's ~15 GB in use. The earlier version loaded the full model in bf16, which is what ran out of room.

**No Hugging Face account or token needed.** It downloads `unsloth/Llama-3.2-3B-Instruct`, an ungated re-upload of Meta's `Llama-3.2-3B-Instruct` weights (Meta's own repo is gated). The Llama 3.2 Community License still applies to the result.

**Before you start**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Have `train.jsonl` and `heldout.jsonl` (from the project's `finetune/` folder) ready to upload in step 3.
3. Run the cells top to bottom -- **Runtime -> Run all** is easiest. After any restart or reconnect, start again from the top: every cell depends on the ones above it.
4. Keep this tab open until the end. If Colab disconnects, everything on its machine is lost.

Rough time: ~5 min setup, ~15-25 min training, ~10 min converting.

## 1. Install dependencies

In [1]:
!pip install -q -U transformers accelerate peft datasets bitsandbytes sentencepiece
# peft checks for torchao at import time and rejects the older version Colab ships; we don't use it, so remove it.
!pip uninstall -y -q torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.7 MB/s eta 0:00:00


## 2. Check the GPU
If this fails, the runtime isn't on a GPU (or a CPU-only torch got installed) -- fix that before going on.

In [2]:
import torch

assert torch.cuda.is_available() and "+cpu" not in torch.__version__, (
    "No GPU detected, or a CPU-only torch is installed. Runtime -> Change runtime type -> T4 GPU, "
    "then Runtime -> Disconnect and delete runtime, reconnect, and Run all from the top."
)
props = torch.cuda.get_device_properties(0)
print(f"torch {torch.__version__} | {props.name} | {props.total_memory / 1e9:.1f} GB")

torch 2.11.0+cu128 | Tesla T4 | 15.6 GB


## 3. Upload the training data
Select **exactly two files**: `train.jsonl` and `heldout.jsonl`. (Not `training_data.json` -- that's the human-editable source, not the file the notebook reads.)

In [3]:
import json
import re
from google.colab import files

uploaded = files.upload()

def pick(base):
    # Match the name exactly (allowing Colab's " (1)" suffix on re-uploads), so that e.g.
    # training_data.json can never be mistaken for train.jsonl.
    pattern = re.compile(rf"^{re.escape(base)}( \(\d+\))?\.jsonl$")
    names = [n for n in uploaded if pattern.match(n)]
    assert names, (
        f"No {base}.jsonl among the uploaded files {list(uploaded)}. "
        "Re-run this cell and select train.jsonl and heldout.jsonl (not training_data.json)."
    )
    return names[-1]

def read_jsonl(name):
    rows = []
    for line_no, line in enumerate(uploaded[name].decode("utf-8").splitlines(), 1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except ValueError:
            raise ValueError(
                f"{name!r}, line {line_no}, isn't a JSON object -- this must be the original .jsonl "
                "file (one JSON object per line), not a pretty-printed or edited copy."
            ) from None
    return rows

train_rows = read_jsonl(pick("train"))
heldout_rows = read_jsonl(pick("heldout"))
assert all("prompt" in r and "completion" in r for r in train_rows), "train.jsonl rows need prompt + completion"
assert all("prompt" in r and "expect" in r for r in heldout_rows), "heldout.jsonl rows need prompt + expect"
print(len(train_rows), "training rows,", len(heldout_rows), "held-out prompts")

Saving heldout.jsonl to heldout.jsonl
Saving ollama_prompt.py to ollama_prompt.py
Saving README.md to README.md
Saving train.jsonl to train.jsonl
Saving training_data.json to training_data.json
95 training rows, 8 held-out prompts


## 4. Load the tokenizer and encode the data

Each row is `prompt` (exactly what Ollama feeds the model) + `completion` (what we want it to write next, ending in `<|eot_id|>`). The loss is applied to the completion only, so the model learns *what to answer*, not to memorize the long system prompt.

In [4]:
from transformers import AutoTokenizer

MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"   # ungated copy of meta-llama/Llama-3.2-3B-Instruct
# If a T4 still runs out of memory later, switch to "unsloth/Llama-3.2-1B-Instruct" (same prompt format), restart, Run all.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
EOT_ID = tokenizer.convert_tokens_to_ids("<|eot_id|>")
print("BOS id:", tokenizer.bos_token_id, "| EOT id:", EOT_ID)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

BOS id: 128000 | EOT id: 128009


In [5]:
def encode_row(row):
    prompt_ids = tokenizer(row["prompt"], add_special_tokens=True)["input_ids"]
    completion_ids = tokenizer(row["completion"], add_special_tokens=False)["input_ids"]
    assert prompt_ids[0] == tokenizer.bos_token_id, "prompt should start with the BOS token"
    assert completion_ids[-1] == EOT_ID, "completion should end with <|eot_id|>"
    return {
        "input_ids": prompt_ids + completion_ids,
        "attention_mask": [1] * (len(prompt_ids) + len(completion_ids)),
        "labels": [-100] * len(prompt_ids) + completion_ids,
    }

encoded = [encode_row(r) for r in train_rows]
lengths = [len(e["input_ids"]) for e in encoded]
trained_tokens = sum(sum(t != -100 for t in e["labels"]) for e in encoded)
print(f"{len(encoded)} rows | longest {max(lengths)} tokens | tokens the loss is computed on: {trained_tokens:,}")

95 rows | longest 2891 tokens | tokens the loss is computed on: 6,075


In [6]:
from datasets import Dataset

train_dataset = Dataset.from_list(encoded)
print(train_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 95
})


## 5. Load the model in 4-bit

In [7]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

def load_model(**kwargs):
    # Newer transformers renamed torch_dtype -> dtype; older ones only know torch_dtype.
    try:
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, **kwargs)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, **kwargs)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = load_model(quantization_config=bnb_config, device_map={"": 0})
model.config.use_cache = False
print(f"Loaded. GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Loaded. GPU memory in use: 2.2 GB


## 6. Add the LoRA adapter
Only a small set of extra weights (about 1% of the model) is trained; the 4-bit base stays frozen.

In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
# fp16 mixed-precision training needs the trainable (LoRA) weights in fp32.
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## 7. Train

95 rows x 4 epochs is a small run: watch the loss trend clearly downward, but don't expect it to hit zero (that would mean memorizing the exact examples). If this cell fails with **CUDA out of memory**, switch `MODEL_ID` in step 4 to the 1B model, restart the runtime, and Run all.

In [9]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="uae-finance-bot-lora",
    num_train_epochs=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,   # (warmup_ratio was removed in transformers 5)
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    report_to="none",
    seed=42,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
trainer.train()
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

Step,Training Loss
5,0.978221
10,0.566028
15,0.545291
20,0.607198
25,0.365213
30,0.244508
35,0.232685
40,0.266019
45,0.295610
50,0.222506


Peak GPU memory: 9.7 GB


## 8. Check it on requests it never trained on

Eight held-out requests: five should produce a tool call (a bare JSON `{"name": ...}` reply), three should produce a plain-text answer. If most **PASS**, the fine-tune took. If most FAIL, stop here -- converting and downloading a model that didn't learn would waste your time; tell me what you see instead.

In [10]:
import json

def first_json(text):
    text = text.strip()
    if not text.startswith("{"):
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(text)
        return obj
    except ValueError:
        return None

def run_heldout(m, label):
    m.eval()
    passed = 0
    print(f"=== held-out check: {label} ===")
    for row in heldout_rows:
        inputs = tokenizer(row["prompt"], return_tensors="pt", add_special_tokens=True).to(m.device)
        with torch.no_grad():
            out = m.generate(
                **inputs,
                max_new_tokens=350,
                do_sample=False,
                use_cache=True,
                eos_token_id=[EOT_ID],
                pad_token_id=tokenizer.pad_token_id,
            )
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        call = first_json(reply)
        if row["expect"] == "text":
            ok = bool(reply) and call is None
        else:
            ok = call is not None and call.get("name") == row["expect"]
        passed += ok
        print(f"[{'PASS' if ok else 'FAIL'}] {row['id']}  (expected: {row['expect']})")
        print("        ->", reply[:260].replace("\n", " "))
    print(f"{passed}/{len(heldout_rows)} passed")
    return passed

run_heldout(model, "after training (4-bit)")

=== held-out check: after training (4-bit) ===


[transformers] Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[PASS] h_vat_vs_freelancer_chart  (expected: render_chart)
        -> {"name": "render_chart", "parameters": {"chart_type": "bar", "title": "VAT vs Freelancer Corporate Tax Thresholds", "labels": ["VAT registration threshold", "Freelancer corporate tax threshold"], "series_1_label": "AED million", "series_1_values": [5, 1]}}


[transformers] Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[PASS] h_leave_bar  (expected: render_chart)
        -> {"name": "render_chart", "parameters": {"chart_type": "bar", "title": "UAE Annual Leave Days", "labels": ["6-12 months service", "1+ year service"], "series_1_label": "Days per year", "series_1_values": [24, 30]}}


[transformers] Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (6 outputs left out)

[PASS] h_text_vat_vs_customs  (expected: text)
        -> Customs duty is a separate charge (commonly 5% of the customs value for most goods, with exceptions like tobacco and alcohol at higher rates, and some goods exempt) levied on imported goods at the point of entry. VAT is applied on top of the customs-inclusive 


[transformers] Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[PASS] h_text_sbr  (expected: text)
        -> Small Business Relief lets a resident taxable person with revenue below AED 3 million per tax period be treated as having no taxable income for that period, subject to conditions and anti-abuse rules. [Federal Decree-Law No. 47 of 2022 -- tax.gov.ae, as of 202
[PASS] h_text_no_data  (expected: text)
        -> I don't have sourced average-salary figures for each emirate in my knowledge base, so I can't chart them without inventing numbers. If you provide the figures, I'll chart them for you.
8/8 passed


8

## 9. Merge the adapter into a full 16-bit model

The adapter was trained against the 4-bit base; merging it into the regular 16-bit weights is the standard way to get a single model file. We then re-run the same check on the merged model to make sure the merge kept the behavior.

In [11]:
import gc, os, shutil
from huggingface_hub import snapshot_download
from peft import PeftModel

ADAPTER_DIR, MERGED_DIR = "uae-finance-bot-lora", "uae-finance-bot-merged"
model.save_pretrained(ADAPTER_DIR)

del model, trainer
gc.collect()
torch.cuda.empty_cache()

base = load_model(device_map={"": 0})   # plain fp16, not 4-bit
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)

# Copy the original config/tokenizer files in so llama.cpp's converter sees a standard Hugging Face layout.
src = snapshot_download(MODEL_ID, allow_patterns=[
    "config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
])
for name in os.listdir(src):
    shutil.copy(os.path.join(src, name), os.path.join(MERGED_DIR, name))

run_heldout(merged, "after merging into fp16")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== held-out check: after merging into fp16 ===


... (11 outputs left out)

[PASS] h_text_vat_vs_customs  (expected: text)
        -> VAT is a federal tax applied to most goods and services at each stage of the supply chain, while customs duty is a separate charge levied on imported goods at the point of entry, collected by each emirate's customs authority. [Source: tax.gov.ae, as of 2026-09


[transformers] Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[PASS] h_text_sbr  (expected: text)
        -> Small Business Relief lets a resident taxable person with revenue below AED 3 million per tax period be treated as having no taxable income for that period, subject to conditions and anti-abuse rules. Eligibility and thresholds are set by Ministerial Decision 
[FAIL] h_text_no_data  (expected: text)
        -> {"name": "render_chart", "parameters": {"chart_type": "bar", "title": "Average Salary by Emirate", "labels": ["Dubai", "Abu Dhabi", "Sharjah", "Ajman", "Ras Al Khaimah", "Fujairah", "Umm Al Quwain"], "series_1_label": "AED per month", "series_1_values": [8, 7,
7/8 passed


7

## 10. Convert to GGUF for Ollama

Ollama runs GGUF files. This converts the merged model to a 16-bit GGUF, then quantizes it to Q4_K_M -- the same size class as your current `llama3.2` (about 2 GB).

We install only what llama.cpp's converter needs: its own requirements file would swap Colab's GPU torch for a CPU build, and it's tested against `transformers==4.57.6`.

In [12]:
!git clone --depth 1 -q https://github.com/ggerganov/llama.cpp
!pip install -q transformers==4.57.6 sentencepiece "protobuf<5"
!python llama.cpp/convert_hf_to_gguf.py uae-finance-bot-merged --outfile uae-finance-bot-f16.gguf --outtype f16

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires protobuf!=5.28.0,!=5.29.0,<8,>=5, but you have protobuf 4.25.9 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompa

In [13]:
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_OPENSSL=OFF -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --target llama-quantize -j 4
!./llama.cpp/build/bin/llama-quantize uae-finance-bot-f16.gguf uae-finance-bot-tuned.gguf Q4_K_M
!ls -lh *.gguf

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.5.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

## 11. Get the file out of Colab

Copying to Google Drive is the most reliable way for a ~2 GB file; download it from Drive in your browser afterwards.

In [17]:
from google.colab import drive

drive.mount("/content/drive")
!cp uae-finance-bot-tuned.gguf /content/drive/MyDrive/
print("Saved: Google Drive -> My Drive -> uae-finance-bot-tuned.gguf")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: Google Drive -> My Drive -> uae-finance-bot-tuned.gguf


In [ ]:
# Alternative: download straight to your computer (can be slow or flaky at ~2 GB -- Drive above is more reliable).
# from google.colab import files
# files.download("uae-finance-bot-tuned.gguf")

## 12. Load it into Ollama (on your own computer, not in Colab)

1. Put `uae-finance-bot-tuned.gguf` in the `MariaBetter` project folder, next to `Modelfile.tuned` (already there -- it carries the Llama 3.2 chat template Ollama needs for tool calling).
2. In a terminal in that folder: `ollama create uae-finance-bot-tuned -f Modelfile.tuned`
3. In `app.py`, set `DEFAULT_LOCAL_MODEL = "uae-finance-bot-tuned"` (or run `python main_local.py uae-finance-bot-tuned`), restart, and try the requests that used to fail.